# 卤化物固态电解质知识问答系统 - 演示

基于 RAG 架构: 中文提问 → 多语言嵌入检索英文文献 → DeepSeek 生成带引用的答案。

前置条件: 已运行 `src/ingest.py` 和 `src/build_db.py` 完成建库, 并在 `.env` 中配置 `DEEPSEEK_API_KEY`。

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from qa import RAGEngine

engine = RAGEngine(db_path=PROJECT_ROOT / "data" / "chroma_db", top_k=4)
print(f"向量库已加载, 共 {engine.collection.count()} 条记录")

## 单次提问示例

In [ ]:
def ask(question: str):
    result = engine.ask(question)
    print(f"Q: {result['question']}\n")
    print(result["answer"])
    print(f"\n--- 检索到的文献 ---")
    for ref in result["references"]:
        print(f"[{ref['n']}] {ref['title']} ({ref['source']})")
    print(f"(生成耗时 {result['elapsed_s']}s)")

ask("Li3YCl6的室温离子电导率是多少? 不同合成方法有差异吗?")

In [ ]:
ask("氯化物和溴化物电解质的电化学窗口分别是多少?")

## 交互式问答

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

input_box = widgets.Text(
    placeholder="请输入问题, 如: Li3HoBr6的离子电导率是多少?",
    layout=widgets.Layout(width="70%"),
)
ask_btn = widgets.Button(description="提问", button_style="primary")
output_area = widgets.Output()

def on_ask(_):
    question = input_box.value.strip()
    if not question:
        return
    input_box.value = ""
    with output_area:
        print(f"Q: {question}\n检索与生成中...")
    try:
        result = engine.ask(question)
        with output_area:
            clear_output()
            print(f"Q: {question}\n")
            print(result["answer"])
            print(f"\n(生成耗时 {result['elapsed_s']}s)")
    except Exception as e:
        with output_area:
            clear_output()
            print(f"出错了: {e}")

ask_btn.on_click(on_ask)
input_box.on_submit(on_ask)
display(widgets.HBox([input_box, ask_btn]), output_area)